# VisDrone2019-DET EDA

This notebook is a starter exploration workspace for the `aerotrack` detection dataset. It assumes `./scripts/download_visDrone.sh` has already prepared `data/visdrone/VisDrone.yaml` and the YOLO label files.

In [ ]:
from pathlib import Path
import random

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml

DATASET_YAML = Path('../data/visdrone/VisDrone.yaml')
with DATASET_YAML.open() as handle:
    dataset_cfg = yaml.safe_load(handle)

dataset_root = DATASET_YAML.parent
train_images = sorted((dataset_root / dataset_cfg['train']).glob('*'))
val_images = sorted((dataset_root / dataset_cfg['val']).glob('*'))
class_names = dataset_cfg['names']

print(f'Train images: {len(train_images)}')
print(f'Val images: {len(val_images)}')
print(class_names)

In [ ]:
def read_yolo_labels(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        class_id, x_center, y_center, width, height = map(float, line.split())
        rows.append({
            'class_id': int(class_id),
            'x_center': x_center,
            'y_center': y_center,
            'width': width,
            'height': height,
        })
    return rows

sample_image = random.choice(train_images)
sample_label = dataset_root / 'labels' / 'train' / f'{sample_image.stem}.txt'
labels = read_yolo_labels(sample_label)
pd.DataFrame(labels).head()

In [ ]:
image = cv2.cvtColor(cv2.imread(str(sample_image)), cv2.COLOR_BGR2RGB)
height, width = image.shape[:2]

for row in labels:
    x_center = row['x_center'] * width
    y_center = row['y_center'] * height
    box_width = row['width'] * width
    box_height = row['height'] * height
    x1 = int(x_center - box_width / 2)
    y1 = int(y_center - box_height / 2)
    x2 = int(x_center + box_width / 2)
    y2 = int(y_center + box_height / 2)
    cv2.rectangle(image, (x1, y1), (x2, y2), (255, 80, 80), 2)
    cv2.putText(image, class_names[row['class_id']], (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 80, 80), 2)

plt.figure(figsize=(14, 10))
plt.imshow(image)
plt.axis('off')
plt.show()